In [ ]:
import os
import sys
import importlib
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive già montato, salto il mount.")

sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
importlib.invalidate_caches()

import common
importlib.reload(common)
from common import *

print(f"Device attivo: {device}")
print(paths)
log('Notebook avviato correttamente')

Mounted at /content/drive
Device attivo: cpu
[2026-09-05 16:51] Notebook avviato


In [ ]:
import os
import pandas as pd

# --- 1. Carica il CSV di calibrazione (fonte di verità per classe/esperimento) ---
calib_path = os.path.join(paths['annotations'], 'calibrazione.csv')
df_calib = pd.read_csv(calib_path)

totale_immagini = len(df_calib)

# --- 2. Riepilogo per classe: conteggio, nm/pixel, esperimenti coinvolti ---
conteggio_classi = df_calib.groupby('classe_calibrazione').agg(
    n_immagini=('filename', 'count'),
    nm_per_pixel=('nm_per_pixel', 'first'),
    esperimenti_coinvolti=('esperimento', lambda x: ', '.join(sorted(x.unique())))
).sort_values('nm_per_pixel').reset_index()

# segnala visivamente le classi con pochi campioni (utile per lo split futuro)
conteggio_classi['attenzione_pochi_dati'] = conteggio_classi['n_immagini'] < 5

# --- 3. Breakdown incrociato classe x esperimento (utile per vedere sbilanciamenti) ---
pivot_esperimento = df_calib.pivot_table(
    index='classe_calibrazione', columns='esperimento',
    values='filename', aggfunc='count', fill_value=0
)

# --- 4. Stampa riepilogo a schermo ---
print(f"Totale immagini nel dataset: {totale_immagini}")
print(f"Numero di classi di calibrazione: {len(conteggio_classi)}\n")
print(conteggio_classi)

Totale immagini nel dataset: 144
Numero di classi di calibrazione: 10

  classe_calibrazione  n_immagini  nm_per_pixel  \
0            classe_1           7         0.450   
1            classe_2           9         0.560   
2            classe_3          17         0.670   
3            classe_4          39         0.845   
4            classe_5          43         1.125   
5            classe_6           6         1.405   
6            classe_7          17         1.680   
7            classe_8           2         2.110   
8            classe_9           2         2.820   
9           classe_10           2         5.630   

                               esperimenti_coinvolti  attenzione_pochi_dati  
0                      28_07_26_U87WT_Immunogold_Ida                  False  
1  06_08_26_U87KO_Immunogold_Ida, 09_07_26_U87WT_...                  False  
2  06_08_26_U87KO_Immunogold_Ida, 09_07_26_U87WT_...                  False  
3  06_08_26_U87KO_Immunogold_Ida, 09_07_26_U87WT_...   